# Stacked model (~85% accuracy)

Combines three kinds of signal from `features_and_embeddings.csv` + `gpt2_embeddings.csv`:

1. **Handcrafted features** — linguistic / storytelling features already in the features file  
2. **GPT-2 embeddings** — 768-d mean final-layer vectors  
3. **Word & character TF-IDF** — patterns learned from the raw transcript text  

Each block is a small function. The final stacked logistic regression is what reaches ~85%.

In [10]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import StackingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import (
    GridSearchCV,
    RepeatedStratifiedKFold,
    StratifiedKFold,
    cross_val_predict,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

DATA = Path("../data")

## 1. Load data

In [11]:
def load_data(
    features_path=DATA / "features_and_embeddings.csv",
    embeddings_path=DATA / "gpt2_embeddings.csv",
    meta_path=None,
):
    """Load features, embeddings, and optional metadata. Align on row order / video_id."""
    feats = pd.read_csv(features_path)
    emb = pd.read_csv(embeddings_path)

    # Target
    if "y" not in feats.columns and "label" in feats.columns:
        feats["y"] = (feats["label"].astype(str).str.lower() == "ai").astype(int)
    y = feats["y"].astype(int)

    # Handcrafted numeric columns (everything except text / ids / target)
    skip = {"transcript", "y", "label", "video_id", "url", "title", "source_query",
            "channel", "theme", "status", "language"}
    handcrafted = [c for c in feats.columns if c not in skip and pd.api.types.is_numeric_dtype(feats[c])]

    # Embeddings
    emb_cols = [c for c in emb.columns if c.startswith("gpt2_emb_")]

    # Align embeddings to features
    if "video_id" in feats.columns and "video_id" in emb.columns:
        emb = emb.set_index("video_id").reindex(feats["video_id"]).reset_index(drop=True)
    else:
        # Assume same row order (as produced by embeddings_generation.py)
        assert len(emb) == len(feats), "features and embeddings have different lengths"

    # Transcript text for TF-IDF
    if "transcript" in feats.columns:
        text = feats["transcript"].astype(str)
    else:
        # Try metadata file
        if meta_path is None:
            for candidate in [DATA / "transcripts_filtered.csv", DATA / "transcripts.csv"]:
                if candidate.exists():
                    meta_path = candidate
                    break
        if meta_path is None or not Path(meta_path).exists():
            raise FileNotFoundError("Need a transcript column or a transcripts CSV for TF-IDF.")
        meta = pd.read_csv(meta_path)
        if "video_id" in feats.columns and "video_id" in meta.columns:
            meta = meta.set_index("video_id").reindex(feats["video_id"])
            text = meta["transcript"].astype(str).values
        else:
            text = meta["transcript"].astype(str)

    X = pd.concat(
        [
            feats[handcrafted].reset_index(drop=True),
            emb[emb_cols].reset_index(drop=True),
            pd.Series(text, name="text").reset_index(drop=True),
        ],
        axis=1,
    )

    # Optional theme for leave-one-theme-out
    theme = None
    if meta_path is not None and Path(meta_path).exists():
        meta = pd.read_csv(meta_path)
        if "theme" in meta.columns and "video_id" in meta.columns and "video_id" in feats.columns:
            theme = meta.set_index("video_id").reindex(feats["video_id"])["theme"].values
        elif "theme" in meta.columns and len(meta) == len(feats):
            theme = meta["theme"].values

    print(
        f"{len(y)} transcripts | handcrafted: {len(handcrafted)} | "
        f"embedding: {len(emb_cols)} | AI={int(y.sum())} human={int((y == 0).sum())}"
    )
    return X, y, handcrafted, emb_cols, theme

In [12]:
X, y, HANDCRAFTED, EMBEDDING, theme = load_data()

364 transcripts | handcrafted: 119 | embedding: 768 | AI=172 human=192


## 2. Base learners

In [13]:
def tuned_lr(cols, cv=None):
    """Logistic regression on a fixed list of numeric columns, with C tuned by inner CV."""
    if cv is None:
        cv = StratifiedKFold(5, shuffle=True, random_state=0)
    pipe = make_pipeline(
        ColumnTransformer([("cols", "passthrough", cols)]),
        StandardScaler(),
        LogisticRegression(max_iter=5000),
    )
    return GridSearchCV(
        pipe,
        {"logisticregression__C": [0.003, 0.01, 0.03, 0.1, 0.3, 1]},
        cv=cv,
        scoring="roc_auc",
    )


def text_lr(analyzer, ngrams, min_df, C=5):
    """TF-IDF + logistic regression on the 'text' column."""
    vec = TfidfVectorizer(
        analyzer=analyzer, ngram_range=ngrams, min_df=min_df, sublinear_tf=True
    )
    return make_pipeline(
        ColumnTransformer([("text", vec, "text")]),
        LogisticRegression(C=C, max_iter=5000),
    )


def build_stacked_model(handcrafted_cols, embedding_cols):
    """Full stacked model: handcrafted + GPT-2 embedding + word/char TF-IDF."""
    parts = [
        ("handcrafted", tuned_lr(handcrafted_cols)),
        ("gpt2_embedding", tuned_lr(embedding_cols)),
        ("word_patterns", text_lr("word", (1, 2), 2)),
        ("char_patterns", text_lr("char_wb", (2, 5), 3)),
    ]
    return StackingClassifier(
        parts,
        final_estimator=LogisticRegression(max_iter=5000),
        cv=StratifiedKFold(5, shuffle=True, random_state=1),
        stack_method="predict_proba",
    )

## 3. Compare setups on repeated stratified CV

In [14]:
def evaluate_setups(X, y, handcrafted_cols, embedding_cols, n_splits=5, n_repeats=4):
    """Run original / simple / stacked setups on the same RepeatedStratifiedKFold splits."""
    # "Original" = first ~36 basic features if we can identify them, else all handcrafted
    basic = [c for c in handcrafted_cols if not (
        c.endswith("_per100") or c.startswith("gpt2_") or c.startswith("gltr_")
        or c in ("climax_position", "conflict_position", "plot_bombs", "topic_jump_share",
                 "stakes_raises", "tempo_change", "monologue_share", "imperative_share",
                 "hook_opening", "moral_ending", "cta_flag")
    )]
    if len(basic) < 10:
        basic = handcrafted_cols[:36]

    simple_extra = [c for c in [
        "discourse_markers_per100", "casual_verbs_per100", "slang_per100",
        "subordinators_per100", "time_connectives_per100", "reveal_words_per100",
        "absolutist_per100", "kinship_per100", "future_modals_per100",
        "love_talk_per100", "apology_per100",
    ] if c in handcrafted_cols]

    setups = {
        "original features (LogReg)": lambda: tuned_lr(basic),
        "simple: original + 11 families": lambda: tuned_lr(basic + simple_extra),
        "max version (stacked)": lambda: build_stacked_model(handcrafted_cols, embedding_cols),
    }

    splits = list(
        RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=42).split(X, y)
    )
    scores = {}
    for name, make in setups.items():
        acc, auc = [], []
        for tr, te in splits:
            proba = make().fit(X.iloc[tr], y.iloc[tr]).predict_proba(X.iloc[te])[:, 1]
            acc.append(accuracy_score(y.iloc[te], proba >= 0.5))
            auc.append(roc_auc_score(y.iloc[te], proba))
        scores[name] = np.array(acc)
        print(f"{name:36s}  accuracy {np.mean(acc):.3f} (+/- {np.std(acc):.3f})   AUC {np.mean(auc):.3f}")

    base = scores["original features (LogReg)"]
    for name in list(setups)[1:]:
        print(
            f"{name} beats original on {np.mean(scores[name] > base):.0%} of splits, "
            f"loses on {np.mean(scores[name] < base):.0%}"
        )
    return scores, setups

In [15]:
scores, SETUPS = evaluate_setups(X, y, HANDCRAFTED, EMBEDDING)

original features (LogReg)            accuracy 0.742 (+/- 0.043)   AUC 0.813
simple: original + 11 families        accuracy 0.777 (+/- 0.050)   AUC 0.856
max version (stacked)                 accuracy 0.836 (+/- 0.042)   AUC 0.913
simple: original + 11 families beats original on 80% of splits, loses on 15%
max version (stacked) beats original on 95% of splits, loses on 0%


## 4. Leave-one-theme-out (topic generalisation)

In [17]:
def leave_one_theme_out(X, y, theme, setups, names=None):
    """Train on all themes but one; test on the held-out theme."""
    if theme is None:
        print("No theme column available — skipping leave-one-theme-out.")
        return
    if names is None:
        names = ["original features (LogReg)", "max version (stacked)"]

    theme = np.asarray(theme)
    for name in names:
        if name not in setups:
            continue
        pred = np.zeros(len(y))
        for th in np.unique(theme):
            tr, te = np.where(theme != th)[0], np.where(theme == th)[0]
            if len(te) == 0 or len(np.unique(y.iloc[tr])) < 2:
                continue
            pred[te] = setups[name]().fit(X.iloc[tr], y.iloc[tr]).predict_proba(X.iloc[te])[:, 1]
        per_theme = {
            th: round(accuracy_score(y[theme == th], pred[theme == th] >= 0.5), 2)
            for th in np.unique(theme)
            if (theme == th).sum() > 0
        }
        print(
            f"{name:36s}  unseen-theme acc {accuracy_score(y, pred >= 0.5):.3f}  "
            f"per theme: {per_theme}"
        )

In [18]:
leave_one_theme_out(X, y, theme, SETUPS)

No theme column available — skipping leave-one-theme-out.


## 5. Classification report (stacked model, out-of-fold)

In [19]:
def oof_report(X, y, handcrafted_cols, embedding_cols):
    """5-fold out-of-fold predictions for the stacked model + classification report."""
    model = build_stacked_model(handcrafted_cols, embedding_cols)
    oof = cross_val_predict(
        model, X, y,
        cv=StratifiedKFold(5, shuffle=True, random_state=42),
        method="predict_proba",
    )[:, 1]
    print(classification_report(y, oof >= 0.5, target_names=["human", "AI"]))
    return oof

In [20]:
oof = oof_report(X, y, HANDCRAFTED, EMBEDDING)

              precision    recall  f1-score   support

       human       0.84      0.90      0.87       192
          AI       0.88      0.81      0.84       172

    accuracy                           0.86       364
   macro avg       0.86      0.85      0.86       364
weighted avg       0.86      0.86      0.86       364



## 6. How much the stack trusts each input

In [21]:
def stacking_weights(X, y, handcrafted_cols, embedding_cols):
    """Fit stacked model on all data and print final-estimator coefficients."""
    model = build_stacked_model(handcrafted_cols, embedding_cols).fit(X, y)
    weights = pd.Series(
        model.final_estimator_.coef_[0],
        index=[name for name, _ in model.estimators],
    )
    print(weights.round(2).to_string())
    return model, weights

In [22]:
final_model, weights = stacking_weights(X, y, HANDCRAFTED, EMBEDDING)

handcrafted       1.82
gpt2_embedding    2.26
word_patterns     2.33
char_patterns     2.45
